In [ ]:
#CODE FOR IRREGULAR PATTERN TEXTILES

import torch
from PIL import Image, ImageFilter, ImageDraw
from diffusers import StableDiffusionInpaintPipeline, AutoencoderKL
from torch.utils.tensorboard import SummaryWriter
import datetime

In [ ]:
BASE_SIZE = 1024
INNER_SIZE = 512
OFFSET = (BASE_SIZE - INNER_SIZE) // 2 

state = {
    "shift": (0, 0), # states for the unrolling
}

In [ ]:
# Convolutional padding function
def patch_conv_circular(model):
    for layer in model.modules():
        if isinstance(layer, torch.nn.Conv2d):
            layer.padding_mode = 'circular'

In [ ]:
# Creation of the visual image for the IP-Adapter
def create_guidance_image(source_img, target_size):
    tiled_bg = Image.new("RGB", (target_size, target_size))
    w, h = source_img.size
    tile_w, tile_h = INNER_SIZE, INNER_SIZE
    source_resized = source_img.resize((tile_w, tile_h))
    for i in range(0, target_size, tile_w):
        for j in range(0, target_size, tile_h):
            tiled_bg.paste(source_resized, (i, j))
    return tiled_bg

In [ ]:
#Callback executed after every denoising step
def callback(pipe, step_index, timestep, callback_kwargs):
    global state
    latents = callback_kwargs["latents"] # noise
    mask = callback_kwargs["mask"] # masck
    image = callback_kwargs["masked_image_latents"] # input image
    num_steps = len(pipe.scheduler.timesteps) # total steps

    #Circular padding activation
    if step_index == int(num_steps * 0.15):
        print(f"Circular padding activation: step {step_index}")
        patch_conv_circular(pipe.unet)      
        
    # Unrolling
    if state["shift"] != (0, 0):
        sx, sy = state["shift"]

        latents = torch.roll(latents, shifts=(-sy, -sx), dims=(-2, -1))
        mask = torch.roll(mask, shifts=(-sy, -sx), dims=(-2, -1))
        image = torch.roll(image, shifts=(-sy, -sx), dims=(-2, -1))

    # Rolling
    if step_index < num_steps - 1:

        H = latents.shape[-2] 
        W = latents.shape[-1]

        start_sigma = W / 8.0
        end_sigma = 0.5

        progress = (step_index - (num_steps * 0.4)) / (num_steps * 0.6)    
        std = start_sigma * (1 - progress) + end_sigma

        if step_index < int(num_steps * 0.4): # fino al 40%
            mean = W * 0.75
            
        else:
           mean = W * 0.25

        temp_dx = torch.normal(mean=mean, std=std, size=(1,)).item()
        temp_dy = torch.normal(mean=mean, std=std, size=(1,)).item()

        if step_index >= int(num_steps * 0.9):
            dx = 0
            dy = 0
            print(f"Step {step_index}: details definition.")
        else:
            dx = int(round(temp_dx)) % W
            dy = int(round(temp_dy)) % H

        latents = torch.roll(latents, shifts=(dy, dx), dims=(-2, -1))
        mask = torch.roll(mask, shifts=(dy, dx), dims=(-2, -1))
        image = torch.roll(image, shifts=(dy, dx), dims=(-2, -1))
    
        state["shift"] = (dx, dy)
    else:
        state["shift"] = (0, 0)

    callback_kwargs["latents"] = latents
    callback_kwargs["mask"] = mask
    callback_kwargs["masked_image_latents"] = image

    return callback_kwargs

In [ ]:
device = "cuda"
model_id = "sd-legacy/stable-diffusion-inpainting"
adapter_id = "h94/IP-Adapter"

vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse",
)

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_id, 
    use_safetensors=False,
    safety_checker=None,
    requires_safety_checker=False,
    vae = vae
)

from diffusers import DDIMScheduler
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

pipe.load_ip_adapter(adapter_id, subfolder="models", weight_name="ip-adapter_sd15.bin")

# IP-Adapter scale
pipe.set_ip_adapter_scale(0.5)

In [ ]:

mask = Image.new('L', (BASE_SIZE, BASE_SIZE), 255)

seed_mask = Image.new('L', (INNER_SIZE, INNER_SIZE), 0)
mask.paste(seed_mask, (OFFSET, OFFSET))

try:
    input_image   = Image.open("Texture/image.png").convert("RGB")
    width, height = input_image.size

    left = (width - INNER_SIZE) / 2
    top = (height - INNER_SIZE) / 2
    right = (width + INNER_SIZE) / 2
    bottom = (height + INNER_SIZE) / 2
    small_texture = input_image.crop((left, top, right, bottom))
except:
    print("Image not found.")
    small_texture = Image.new("RGB", (INNER_SIZE, INNER_SIZE), "red")


canvas = Image.new('RGB', (BASE_SIZE, BASE_SIZE), "white")
canvas.paste(small_texture, (OFFSET, OFFSET))

display(mask)
display(small_texture)

In [ ]:
bg_hint = Image.new("RGB", (BASE_SIZE, BASE_SIZE), "white") 
ip_adapter_image = create_guidance_image(small_texture, BASE_SIZE) 

display(ip_adapter_image)

In [ ]:
state["shift"] = (0, 0)
prompt = "texture, close up, hyperrealistic, top down view, high definition"
negative_prompt = "cartoon, drawing, painting, illustration, blurry, smooth, deformed, noisy"

seed = 1 
generator = torch.Generator(device="cpu").manual_seed(1)

pipe.enable_model_cpu_offload()

final_output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canvas, 
    mask_image=mask,
    generator=generator,
    mask_blur=0,
    ip_adapter_image=ip_adapter_image,
    num_inference_steps=100,
    guidance_scale=7.5,
    height=BASE_SIZE,
    width=BASE_SIZE,
    strength=1.0,
    callback_on_step_end=callback,
    callback_on_step_end_tensor_inputs=["latents", "mask", "masked_image_latents"] # input callback
).images[0]

display(final_output)

final_output.save("result.png")

# To underline the seed
draw = ImageDraw.Draw(final_output) 
rect_coords = [OFFSET, OFFSET, OFFSET + INNER_SIZE, OFFSET + INNER_SIZE]
draw.rectangle(rect_coords, outline="green", width=4)

display(final_output)

final_output.save("result_with_corner.png")